In [1]:
import pandas as pd
import datetime as dt
from pathlib import Path

In [2]:

# Load raw CSV files and organize tables for cleaning.
dirty_dir = Path("funnel_data/dirty")
clean_dir = Path("funnel_data/cleaned")
clean_dir.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(dirty_dir / "customers.csv")
products = pd.read_csv(dirty_dir / "products.csv")
events = pd.read_csv(dirty_dir / "events.csv")
orders = pd.read_csv(dirty_dir / "orders.csv")
order_items = pd.read_csv(dirty_dir / "order_items.csv")

tables = {
    "customers": customers,
    "products": products,
    "events": events,
    "orders": orders,
    "order_items": order_items,
}


In [3]:
# Inspect table dimensions and exact duplicate rows.
for name, df in tables.items():
    print(f"{name}: {df.shape}")
    print(f"{name}: {df.duplicated().sum():,} exact duplicate rows")

customers: (50000, 4)
customers: 0 exact duplicate rows
products: (500, 5)
products: 0 exact duplicate rows
events: (1059422, 12)
events: 1,023 exact duplicate rows
orders: (26052, 7)
orders: 26 exact duplicate rows
order_items: (26052, 6)
order_items: 26 exact duplicate rows


In [4]:
# Remove exact duplicate records from event, order, and order-item tables.
for name in ["events", "orders", "order_items"]:
    before = len(tables[name])

    tables[name] = tables[name].drop_duplicates().copy()

    after = len(tables[name])

    print(f"{name}: removed {before - after:,} rows "
            f"({before:,} -> {after:,})")
for name, df in tables.items():
    print(f"{name}: {df.duplicated().sum():,} exact duplicates")

events: removed 1,023 rows (1,059,422 -> 1,058,399)
orders: removed 26 rows (26,052 -> 26,026)
order_items: removed 26 rows (26,052 -> 26,026)
customers: 0 exact duplicates
products: 0 exact duplicates
events: 0 exact duplicates
orders: 0 exact duplicates
order_items: 0 exact duplicates


In [5]:
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print(df.dtypes)


--- customers ---
customer_id           object
signup_date           object
acquisition_source    object
country               object
dtype: object

--- products ---
product_id     object
category       object
subcategory    object
brand          object
price          object
dtype: object

--- events ---
event_id            object
customer_id         object
session_id          object
event_timestamp     object
event_type          object
product_id          object
device              object
traffic_source      object
category            object
subcategory         object
brand               object
price              float64
dtype: object

--- orders ---
order_id           object
customer_id        object
session_id         object
order_timestamp    object
payment_method     object
order_status       object
total_amount       object
dtype: object

--- order_items ---
order_id         object
product_id       object
unit_price      float64
quantity          int64
discount_pct    float64
li

In [ ]:
# Convert date and timestamp columns.

In [6]:
tables["customers"]["signup_date"] = pd.to_datetime(tables["customers"]["signup_date"], errors="coerce")

In [7]:
tables["events"]["event_timestamp"] = pd.to_datetime(tables["events"]["event_timestamp"], errors="coerce")

In [8]:
tables["orders"]["order_timestamp"] = pd.to_datetime(tables["orders"]["order_timestamp"], errors="coerce")

In [9]:
# Round timestamps to microsecond precision for consistent storage.
timestamp_columns = {
    "customers": ["signup_date"],
    "events": ["event_timestamp"],
    "orders": ["order_timestamp"],
}

for table_name, columns in timestamp_columns.items():
    for column in columns:
        tables[table_name][column] = (
            tables[table_name][column].dt.round("us")
        )

In [10]:
# Convert numeric fields to appropriate types
numeric_columns = {"products": ["price"], 
                   "events": ["price"], 
                   "orders": ["total_amount"], 
                   "order_items": ["unit_price", "quantity", 
                                   "discount_pct", "line_total"]}

for table_name, columns in numeric_columns.items():
    for column in columns:
        tables[table_name][column] = pd.to_numeric(tables[table_name][column], errors="coerce")

In [11]:
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print(df.dtypes)


--- customers ---
customer_id                   object
signup_date           datetime64[ns]
acquisition_source            object
country                       object
dtype: object

--- products ---
product_id      object
category        object
subcategory     object
brand           object
price          float64
dtype: object

--- events ---
event_id                   object
customer_id                object
session_id                 object
event_timestamp    datetime64[ns]
event_type                 object
product_id                 object
device                     object
traffic_source             object
category                   object
subcategory                object
brand                      object
price                     float64
dtype: object

--- orders ---
order_id                   object
customer_id                object
session_id                 object
order_timestamp    datetime64[ns]
payment_method             object
order_status               object
total_amount  

In [12]:
# Normalize categorical text by trimming whitespace and converting to lowercase.
text_columns = {
    "products": ["category", "subcategory", "brand"],
    "events": [
        "event_type",
        "device",
        "traffic_source",
        "category",
        "subcategory",
        "brand",
    ],
    "orders": ["payment_method", "order_status"],
}

for table_name, columns in text_columns.items():
    for column in columns:
        tables[table_name][column] = tables[table_name][column].str.strip().str.lower()

In [13]:
# Define the expected event types and order statuses.
valid_event_types = {
    "view",
    "add_to_cart",
    "checkout",
    "purchase",
}

valid_order_statuses = {
    "completed",
    "cancelled",
    "refunded",
}

In [ ]:
# Identify unexpected categorical values for review.

In [14]:
invalid_events = tables["events"][~tables["events"]["event_type"].isin(valid_event_types)]

In [15]:
invalid_orders = tables["orders"][~tables["orders"]["order_status"].isin(valid_order_statuses)]

In [16]:
print("Unexpected event types:")
print(invalid_events["event_type"].value_counts(dropna=False))

print("\nUnexpected order statuses:")
print(invalid_orders["order_status"].value_counts(dropna=False))

Unexpected event types:
event_type
view_item            1
add_to_basket        1
click                1
purchase_complete    1
unknown              1
Name: count, dtype: int64

Unexpected order statuses:
order_status
pending       1
processing    1
failed        1
shipped       1
Name: count, dtype: int64


In [ ]:
# Replace unexpected event types and order statuses with missing values.

In [17]:
tables["events"]["event_type"] = tables["events"]["event_type"].where(tables["events"]["event_type"].isin(valid_event_types), pd.NA)

In [18]:
tables["orders"]["order_status"] = tables["orders"]["order_status"].where(tables["orders"]["order_status"].isin(valid_order_statuses), pd.NA)

In [19]:
# Identify non-positive product prices and mark them as missing.
invalid_product_prices = tables["products"]["price"] <= 0
print("Non-positive product prices:", invalid_product_prices.sum())

tables["products"].loc[invalid_product_prices, "price"] = pd.NA

Non-positive product prices: 3


In [20]:
print("Event prices below zero:", (tables["events"]["price"] < 0).sum())
print("Order totals below zero:", (tables["orders"]["total_amount"] < 0).sum())
print("Unit prices below zero:", (tables["order_items"]["unit_price"] < 0).sum())
print("Quantities at or below zero:", (tables["order_items"]["quantity"] <= 0).sum())
print("Discounts below zero:", (tables["order_items"]["discount_pct"] < 0).sum())
print("Discounts above 100%:", (tables["order_items"]["discount_pct"] > 1).sum())
print("Line totals below zero:", (tables["order_items"]["line_total"] < 0).sum())

Event prices below zero: 0
Order totals below zero: 0
Unit prices below zero: 0
Quantities at or below zero: 0
Discounts below zero: 0
Discounts above 100%: 0
Line totals below zero: 0


In [21]:
# Review missing values across all tables.
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print(df.isna().sum()[df.isna().sum() > 0])


--- customers ---
Series([], dtype: int64)

--- products ---
brand    15
price     5
dtype: int64

--- events ---
event_timestamp        1
event_type             5
device              5297
traffic_source     10594
dtype: int64

--- orders ---
payment_method    260
order_status        4
total_amount        2
dtype: int64

--- order_items ---
Series([], dtype: int64)


In [22]:
tables["orders"].loc[tables["orders"]["total_amount"].isna(), ["order_id", "customer_id", "total_amount", "order_status"]]

,order_id,customer_id,total_amount,order_status
4,O0000005,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,NaN,completed
5,O0000006,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,NaN,completed


In [23]:
affected_order_ids = tables["orders"].loc[
    tables["orders"]["total_amount"].isna(),
    "order_id"
]

tables["order_items"].loc[
    tables["order_items"]["order_id"].isin(affected_order_ids)
].sort_values(["order_id", "product_id"])

,order_id,product_id,unit_price,quantity,discount_pct,line_total
4,O0000005,P0005,54.61,1,0.1,49.15
5,O0000006,P0203,28.88,1,0.0,28.88


In [24]:
# Reconstruct missing order totals from their associated line-item totals.
reconstructed_totals = tables["order_items"].groupby("order_id")["line_total"].sum()
missing_total_mask = tables["orders"]["total_amount"].isna()

tables["orders"].loc[missing_total_mask, "total_amount"] = tables["orders"].loc[missing_total_mask, "order_id"].map(reconstructed_totals)


In [25]:
print("Missing order totals:", tables["orders"]["total_amount"].isna().sum())

tables["orders"].loc[
    tables["orders"]["order_id"].isin(affected_order_ids),
    ["order_id", "total_amount"]
]

Missing order totals: 0


,order_id,total_amount
4,O0000005,49.15
5,O0000006,28.88


In [26]:
for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n--- {name} ---")
    print(missing if not missing.empty else "No missing values")


--- customers ---
No missing values

--- products ---
brand    15
price     5
dtype: int64

--- events ---
event_timestamp        1
event_type             5
device              5297
traffic_source     10594
dtype: int64

--- orders ---
payment_method    260
order_status        4
dtype: int64

--- order_items ---
No missing values


In [27]:
# Check key fields for missing and duplicate identifiers.
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "events": "event_id",
    "orders": "order_id",
    "order_items": None,
}

for name, key in primary_keys.items():
    if key is None:
        continue

    df = tables[name]

    print(f"\n--- {name} ---")
    print(f"Missing {key}: {df[key].isna().sum()}")
    print(f"Duplicate {key} rows: {df[key].duplicated().sum()}")
    print(f"Unique {key} values: {df[key].nunique()}")


--- customers ---
Missing customer_id: 0
Duplicate customer_id rows: 0
Unique customer_id values: 50000

--- products ---
Missing product_id: 0
Duplicate product_id rows: 0
Unique product_id values: 500

--- events ---
Missing event_id: 0
Duplicate event_id rows: 35
Unique event_id values: 1058364

--- orders ---
Missing order_id: 0
Duplicate order_id rows: 0
Unique order_id values: 26026


In [28]:
# Inspect duplicate event IDs and check for conflicting records.
duplicate_events = tables["events"][
    tables["events"]["event_id"].duplicated(keep=False)
].sort_values("event_id")

duplicate_events.head(20)

,event_id,customer_id,session_id,event_timestamp,event_type,product_id,device,traffic_source,category,subcategory,brand,price
1059033,E1000332,8c2a4ea9-a9e1-4191-893a-c461daa161fb,S235463,2026-05-27 18:59:03.085672,view,P0177,desktop,social,sportswear,activewear,hoka,71.90
1000331,E1000332,8c2a4ea9-a9e1-4191-893a-c461daa161fb,S235463,2026-05-27 18:59:03.085672,view,P0177,NaN,social,sportswear,activewear,hoka,71.90
1058383,E1010849,3041f372-c964-4398-8247-0766858bf143,S237983,2026-05-28 09:58:34.349850,view,P0406,NaN,paid search,home and lifestyle,water bottles,nike,16.43
1010848,E1010849,3041f372-c964-4398-8247-0766858bf143,S237983,2026-05-28 09:58:34.349850,view,P0406,mobile,paid search,home and lifestyle,water bottles,nike,16.43
1030843,E1030844,5f7d2868-9458-40e5-8ce1-e2d5e8bbb18f,S242622,2025-12-11 10:51:56.594724,view,P0076,desktop,social,sportswear,activewear,new balance,77.51
1058390,E1030844,5f7d2868-9458-40e5-8ce1-e2d5e8bbb18f,S242622,2025-12-11 10:51:56.594724,view,P0076,desktop,NaN,sportswear,activewear,new balance,77.51
1054252,E1054253,f08b919b-7697-45db-a19a-57fe9b90b3ea,S248125,2026-04-24 11:41:44.797120,view,P0004,desktop,social,accessories,belts,herschel,17.80
1058872,E1054253,f08b919b-7697-45db-a19a-57fe9b90b3ea,S248125,2026-04-24 11:41:44.797120,view,P0004,desktop,NaN,accessories,belts,herschel,17.80
1055047,E1055048,48bb0083-5052-40eb-9d77-e8c48700a6db,S248306,2025-12-15 14:10:40.554565,view,P0339,desktop,social,accessories,belts,adidas,9.43
1059240,E1055048,48bb0083-5052-40eb-9d77-e8c48700a6db,S248306,2025-12-15 14:10:40.554565,view,P0339,desktop,NaN,accessories,belts,adidas,9.43


In [29]:
duplicate_events = tables["events"][
    tables["events"]["event_id"].duplicated(keep=False)
].copy()

core_columns = [
    "customer_id",
    "session_id",
    "event_timestamp",
    "event_type",
    "product_id",
    "price",
]

core_variation = (
    duplicate_events
    .groupby("event_id")[core_columns]
    .nunique(dropna=False)
)

print("Duplicate event IDs:", len(core_variation))
print(
    "IDs with conflicting core details:",
    (core_variation > 1).any(axis=1).sum()
)

Duplicate event IDs: 35
IDs with conflicting core details: 0


In [30]:
# Keep the most complete record for each event ID.
before = len(tables["events"])

tables["events"] = tables["events"].assign(_missing_count=tables["events"].isna().sum(axis=1)).sort_values("_missing_count").drop_duplicates(subset="event_id", keep="first").drop(columns="_missing_count").copy()


after = len(tables["events"])

print(f"Rows removed: {before - after:,}")
print(f"Rows remaining: {after:,}")
print(
    "Duplicate event IDs remaining:",
    tables["events"]["event_id"].duplicated().sum()
)

Rows removed: 35
Rows remaining: 1,058,364
Duplicate event IDs remaining: 0


In [31]:
duplicate_orders = tables["orders"][
    tables["orders"]["order_id"].duplicated(keep=False)
].sort_values("order_id")

duplicate_orders

,order_id,customer_id,session_id,order_timestamp,payment_method,order_status,total_amount


In [32]:
before = len(tables["orders"])

tables["orders"] = tables["orders"].assign(
        _missing_count=tables["orders"].isna().sum(axis=1)
    ).sort_values("_missing_count").drop_duplicates(subset="order_id", keep="first").drop(columns="_missing_count").copy()


after = len(tables["orders"])

print(f"Rows removed: {before - after:,}")
print(f"Rows remaining: {after:,}")
print(
    "Duplicate order IDs remaining:",
    tables["orders"]["order_id"].duplicated().sum()
)

Rows removed: 0
Rows remaining: 26,026
Duplicate order IDs remaining: 0


In [33]:
# Check that foreign-key values match records in related tables.
relationships = [
    ("events.customer_id", tables["events"], "customer_id",
     tables["customers"], "customer_id"),
    ("events.product_id", tables["events"], "product_id",
     tables["products"], "product_id"),
    ("orders.customer_id", tables["orders"], "customer_id",
     tables["customers"], "customer_id"),
    ("orders.session_id", tables["orders"], "session_id",
     tables["events"], "session_id"),
    ("order_items.order_id", tables["order_items"], "order_id",
     tables["orders"], "order_id"),
    ("order_items.product_id", tables["order_items"], "product_id",
     tables["products"], "product_id"),
]

for label, child_df, child_col, parent_df, parent_col in relationships:
    unmatched = ~child_df[child_col].isin(parent_df[parent_col])
    print(f"{label}: {unmatched.sum():,} unmatched rows")

events.customer_id: 0 unmatched rows
events.product_id: 0 unmatched rows
orders.customer_id: 0 unmatched rows
orders.session_id: 0 unmatched rows
order_items.order_id: 0 unmatched rows
order_items.product_id: 0 unmatched rows


In [34]:
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Exact duplicate rows: {df.duplicated().sum():,}")
    print(f"Total missing values: {df.isna().sum().sum():,}")


--- customers ---
Rows: 50,000
Columns: 4
Exact duplicate rows: 0
Total missing values: 0

--- products ---
Rows: 500
Columns: 5
Exact duplicate rows: 0
Total missing values: 20

--- events ---
Rows: 1,058,364
Columns: 12
Exact duplicate rows: 0
Total missing values: 15,862

--- orders ---
Rows: 26,026
Columns: 7
Exact duplicate rows: 0
Total missing values: 264

--- order_items ---
Rows: 26,026
Columns: 6
Exact duplicate rows: 0
Total missing values: 0


In [35]:
for name, df in tables.items():
    output_path = clean_dir / f"{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {name}: {len(df):,} rows -> {output_path}")

Saved customers: 50,000 rows -> funnel_data/cleaned/customers.csv
Saved products: 500 rows -> funnel_data/cleaned/products.csv
Saved events: 1,058,364 rows -> funnel_data/cleaned/events.csv
Saved orders: 26,026 rows -> funnel_data/cleaned/orders.csv
Saved order_items: 26,026 rows -> funnel_data/cleaned/order_items.csv


In [36]:
DATA_DIR = Path("funnel_data/cleaned")

customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
events = pd.read_csv(DATA_DIR / "events.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

# Restore data types
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"], errors="coerce"
)
products["price"] = pd.to_numeric(products["price"], errors="coerce")
events["event_timestamp"] = pd.to_datetime(
    events["event_timestamp"], errors="coerce"
)
events["price"] = pd.to_numeric(events["price"], errors="coerce")
orders["order_timestamp"] = pd.to_datetime(
    orders["order_timestamp"], errors="coerce"
)
orders["total_amount"] = pd.to_numeric(
    orders["total_amount"], errors="coerce"
)

for col in ["unit_price", "discount_pct", "line_total"]:
    order_items[col] = pd.to_numeric(
        order_items[col], errors="coerce"
    )

# 1. Table-level checks
tables = {
    "customers": customers,
    "products": products,
    "events": events,
    "orders": orders,
    "order_items": order_items,
}

print("=== TABLE VALIDATION ===")

for name, df in tables.items():
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Exact duplicate rows: {df.duplicated().sum():,}")
    print(f"Total missing values: {df.isna().sum().sum():,}")

    missing = df.isna().sum()
    if missing.any():
        print("Missing by column:")
        print(missing[missing > 0])


# 2. Primary-key checks
primary_keys = {
    "customers": (customers, "customer_id"),
    "products": (products, "product_id"),
    "events": (events, "event_id"),
    "orders": (orders, "order_id"),
}

print("\n=== PRIMARY KEY VALIDATION ===")

for name, (df, key) in primary_keys.items():
    print(
        f"{name}.{key}: "
        f"missing={df[key].isna().sum():,}, "
        f"duplicate rows={df[key].duplicated().sum():,}, "
        f"unique={df[key].nunique():,}"
    )


# 3. Foreign-key checks
foreign_keys = [
    ("events.customer_id", events, "customer_id", customers, "customer_id"),
    ("events.product_id", events, "product_id", products, "product_id"),
    ("orders.customer_id", orders, "customer_id", customers, "customer_id"),
    ("orders.session_id", orders, "session_id", events, "session_id"),
    ("order_items.order_id", order_items, "order_id", orders, "order_id"),
    ("order_items.product_id", order_items, "product_id", products, "product_id"),
]

print("\n=== FOREIGN KEY VALIDATION ===")

for label, child, child_col, parent, parent_col in foreign_keys:
    valid_values = set(parent[parent_col].dropna())
    unmatched = (
        child[child_col].notna()
        & ~child[child_col].isin(valid_values)
    ).sum()

    print(f"{label}: {unmatched:,} unmatched rows")


# 4. Allowed categorical values
print("\n=== CATEGORY VALIDATION ===")

valid_event_types = {"view", "add_to_cart", "checkout", "purchase"}
valid_order_statuses = {"completed", "cancelled"}
valid_payment_methods = {"credit_card", "paypal", "debit_card", "gift_card"}

print("\nUnexpected event types:")
print(
    events.loc[
        ~events["event_type"].isin(valid_event_types),
        "event_type"
    ].value_counts(dropna=False)
)

print("\nUnexpected order statuses:")
print(
    orders.loc[
        ~orders["order_status"].isin(valid_order_statuses),
        "order_status"
    ].value_counts(dropna=False)
)

print("\nUnexpected payment methods:")
print(
    orders.loc[
        orders["payment_method"].notna()
        & ~orders["payment_method"].isin(valid_payment_methods),
        "payment_method"
    ].value_counts(dropna=False)
)


# 5. Numeric and timestamp constraints
print("\n=== VALUE VALIDATION ===")

checks = {
    "Product prices below zero": (products["price"] < 0).sum(),
    "Event prices below zero": (events["price"] < 0).sum(),
    "Order totals below zero": (orders["total_amount"] < 0).sum(),
    "Unit prices below zero": (order_items["unit_price"] < 0).sum(),
    "Quantities at or below zero": (order_items["quantity"] <= 0).sum(),
    "Discounts below 0%": (order_items["discount_pct"] < 0).sum(),
    "Discounts above 100%": (order_items["discount_pct"] > 100).sum(),
    "Line totals below zero": (order_items["line_total"] < 0).sum(),
    "Unparseable event timestamps": events["event_timestamp"].isna().sum(),
    "Unparseable order timestamps": orders["order_timestamp"].isna().sum(),
}

for label, count in checks.items():
    print(f"{label}: {count:,}")


# 6. Reconcile order totals against order items
print("\n=== ORDER TOTAL RECONCILIATION ===")

item_totals = (
    order_items.groupby("order_id", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "calculated_total"})
)

reconciliation = orders[["order_id", "total_amount"]].merge(
    item_totals,
    on="order_id",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

reconciliation["difference"] = (
    reconciliation["total_amount"] - reconciliation["calculated_total"]
)

tolerance = 0.01

mismatches = reconciliation.loc[
    reconciliation["_merge"].ne("both")
    | reconciliation["difference"].abs().gt(tolerance)
]

print(f"Orders checked: {len(reconciliation):,}")
print(f"Orders with missing/multiple-side records or mismatched totals: {len(mismatches):,}")

if not mismatches.empty:
    print(mismatches.head(20).to_string(index=False))


# 7. Final summary
print("\n=== VALIDATION COMPLETE ===")

=== TABLE VALIDATION ===

customers
Rows: 50,000
Columns: 4
Exact duplicate rows: 0
Total missing values: 0

products
Rows: 500
Columns: 5
Exact duplicate rows: 0
Total missing values: 20
Missing by column:
brand    15
price     5
dtype: int64

events
Rows: 1,058,364
Columns: 12
Exact duplicate rows: 0
Total missing values: 15,862
Missing by column:
event_timestamp        1
event_type             5
device              5288
traffic_source     10568
dtype: int64

orders
Rows: 26,026
Columns: 7
Exact duplicate rows: 0
Total missing values: 264
Missing by column:
payment_method    260
order_status        4
dtype: int64

order_items
Rows: 26,026
Columns: 6
Exact duplicate rows: 0
Total missing values: 0

=== PRIMARY KEY VALIDATION ===
customers.customer_id: missing=0, duplicate rows=0, unique=50,000
products.product_id: missing=0, duplicate rows=0, unique=500
events.event_id: missing=0, duplicate rows=0, unique=1,058,364
orders.order_id: missing=0, duplicate rows=0, unique=26,026

=== FOREI